# 04 — Customer Segmentation: RFM Methodology & Prototype (Issue #5)

This notebook is limited to **Issue #5**:

- validate the agreed RFM definitions and reference-date rule;
- demonstrate percentile-rank R/F/M scoring and rule-based segmentation;
- test required edge cases on deterministic sample data;
- validate the **designed schema** of the future `customer_segments.csv`;
- show that reusable code is ready to run later on `cleaned_retail.csv`.

> **Out of scope:** official full-data segmentation and exporting the production `customer_segments.csv`.

**Core rules**

- Recency = calendar days since the latest valid purchase.
- Frequency = distinct valid non-empty invoices (`nunique(InvoiceNo)`).
- Monetary = sum of valid `Quantity × UnitPrice`.
- Reference date = one calendar day after the latest valid identified-customer purchase.
- Duplicate handling is owned by the upstream cleaning pipeline; RFM audits duplicates but does not remove them.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing src/.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    CUSTOMER_SEGMENTS_COLUMNS,
    audit_rfm_input,
    calculate_rfm,
    prepare_rfm_transactions,
    score_rfm_metric,
)

print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Deterministic prototype sample

The synthetic sample deliberately contains:

- multiple invoices for one customer;
- multiple product lines on the same invoice;
- one-order customers;
- cancelled invoice;
- missing/blank `InvoiceNo`;
- missing `CustomerID`;
- anonymous `Guest`;
- non-positive quantity;
- non-positive unit price;
- exact duplicate rows (audited, not removed by the RFM layer);
- customers with equal raw RFM values.

Using a small deterministic table makes the expected results manually auditable.

In [ ]:
sample = pd.DataFrame([
    ["10001", "A", 1, "2011-12-01 10:00", 10.0, "1"],
    ["10001", "B", 2, "2011-12-01 10:00",  5.0, "1"],
    ["10002", "C", 1, "2011-12-05 10:00", 20.0, "1"],
    ["10003", "A", 1, "2011-12-09 09:00", 10.0, "2"],
    ["10004", "A", 2, "2011-12-07 09:00", 10.0, "3"],
    ["10005", "A", 2, "2011-12-07 09:00", 10.0, "4"],
    ["C10006", "A", -1, "2011-12-08 09:00", 10.0, "5"],
    ["10007", "A", 1, "2011-12-08 09:00", 10.0, None],
    ["10008", "A", 1, "2011-12-08 09:00", 10.0, "Guest"],
    ["10009", "A", 0, "2011-12-08 09:00", 10.0, "6"],
    ["10010", "A", 1, "2011-12-08 09:00", 0.0, "7"],
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
    ["10011", "D", 1, "2011-12-06 09:00", 15.0, "8"],
    [None, "E", 1, "2011-12-08 09:00", 12.0, "9"],
    ["   ", "F", 1, "2011-12-08 09:00", 13.0, "10"],
], columns=[
    "InvoiceNo", "StockCode", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID",
])

sample

## 2. Audit and valid-purchase population

The audit is descriptive. Counts may overlap because one row can violate more than one eligibility rule.
Missing/blank `InvoiceNo` rows are ineligible for RFM. Exact duplicates are reported by the audit but deliberately left untouched here because deduplication belongs to the upstream cleaning pipeline.


In [ ]:
pd.Series(audit_rfm_input(sample), name="rows")

In [ ]:
valid_sample = prepare_rfm_transactions(sample)

valid_sample[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate",
     "UnitPrice", "CustomerID", "Revenue"]
].sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])

## 3. Calculate prototype RFM

The latest valid identified-customer purchase in this sample is **2011-12-09**.

Therefore:

```text
ReferenceDate = 2011-12-10
```

A customer purchasing on 2011-12-09 has `Recency = 1`.

In [ ]:
prototype_rfm, metadata = calculate_rfm(sample, return_metadata=True)

pd.Series({
    "valid_transaction_rows": metadata["valid_transaction_rows"],
    "customers": metadata["customers"],
    "invoices": metadata["invoices"],
    "last_valid_purchase": metadata["last_valid_purchase"],
    "reference_date": metadata["reference_date"],
})

In [ ]:
prototype_rfm.sort_values("CustomerID")

## 4. Verify RFM definitions and required edge cases

In [ ]:
c1 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("1")].iloc[0]
c2 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("2")].iloc[0]
c3 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("3")].iloc[0]
c4 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("4")].iloc[0]
c8 = prototype_rfm.loc[prototype_rfm["CustomerID"].eq("8")].iloc[0]

assert metadata["reference_date"] == pd.Timestamp("2011-12-10")

# Customer 1 has 3 product rows but only 2 distinct invoices.
assert c1["Frequency"] == 2

# Monetary = 1*10 + 2*5 + 1*20 = 40.
assert c1["Monetary"] == 40.0

# Latest valid date 2011-12-09 -> Recency = 1.
assert c2["Recency"] == 1

# Invalid / anonymous rows do not enter customer RFM.
assert set(prototype_rfm["CustomerID"]) == {"1", "2", "3", "4", "8"}

# RFM does not silently deduplicate upstream data. The duplicate rows share
# one InvoiceNo, so Frequency remains 1, while both rows contribute to Monetary.
# A production cleaned_retail.csv is expected to have already applied the
# pipeline's agreed duplicate rule before reaching this layer.
assert c8["Frequency"] == 1
assert c8["Monetary"] == 30.0

# Missing/blank InvoiceNo rows must not enter RFM.
assert "9" not in set(prototype_rfm["CustomerID"])
assert "10" not in set(prototype_rfm["CustomerID"])

# Equal raw RFM values receive equal R/F/M scores.
assert tuple(c3[["Recency", "Frequency", "Monetary"]]) == tuple(
    c4[["Recency", "Frequency", "Monetary"]]
)
assert tuple(c3[["R_score", "F_score", "M_score"]]) == tuple(
    c4[["R_score", "F_score", "M_score"]]
)

print("RFM definition and edge-case checks passed.")

## 5. Verify percentile-rank scoring

The agreed scoring method is:

```python
percentile_rank = metric.rank(method="average", pct=True)
score = ceil(percentile_rank * 5)
```

For Frequency and Monetary, higher values receive higher scores.

For Recency, the score is reversed:

```text
R_score = 6 - ceil(percentile_rank(Recency) × 5)
```

`method="average"` ensures equal raw values receive the same percentile rank and the same score. The method also works when fewer than five distinct values exist.

In [ ]:
tie_demo = pd.Series([1, 1, 1, 2, 2, 3], dtype=float, name="Frequency")
tie_scores, tie_meta = score_rfm_metric(
    tie_demo,
    higher_is_better=True,
)

tie_result = pd.DataFrame({
    "Frequency": tie_demo,
    "F_score": tie_scores,
})

display(tie_result)
pd.Series(tie_meta)

In [ ]:
assert tie_scores.tolist() == [2, 2, 2, 4, 4, 5]
assert tie_scores[tie_demo.eq(1)].nunique() == 1
assert tie_scores[tie_demo.eq(2)].nunique() == 1
assert tie_meta["method"] == "percentile_rank_average"
assert tie_meta["rank_method"] == "average"

single_value_scores, single_meta = score_rfm_metric(
    pd.Series([7, 7, 7], dtype=float),
    higher_is_better=True,
)
assert single_value_scores.tolist() == [4, 4, 4]
assert single_meta["method"] == "percentile_rank_average"

recency_demo = pd.Series([1, 10, 30, 60, 120, 300], dtype=float)
r_scores, _ = score_rfm_metric(
    recency_demo,
    higher_is_better=False,
)
assert r_scores.iloc[0] > r_scores.iloc[-1]

print("Percentile-rank scoring checks passed.")

## 6. Validate the designed `customer_segments.csv` schema

Issue #5 requires the **schema design**, not production CSV export.

The prototype DataFrame must match the agreed future customer-level structure.

The segmentation contract for Issue #5 contains exactly six allowed labels:

```text
Champions
Loyal Customers
At Risk
Potential Loyalists
Hibernating
Others
```

The baseline does not infer customer tenure or first-purchase status from RFM Recency.


In [ ]:
assert prototype_rfm.columns.tolist() == CUSTOMER_SEGMENTS_COLUMNS
assert prototype_rfm["CustomerID"].is_unique
assert prototype_rfm["CustomerID"].notna().all()

assert prototype_rfm["Recency"].ge(1).all()
assert prototype_rfm["Frequency"].ge(1).all()
assert prototype_rfm["Monetary"].gt(0).all()

assert prototype_rfm["R_score"].between(1, 5).all()
assert prototype_rfm["F_score"].between(1, 5).all()
assert prototype_rfm["M_score"].between(1, 5).all()
assert prototype_rfm["RFM_total"].between(3, 15).all()
assert prototype_rfm["RFM_score"].astype(str).str.fullmatch(r"[1-5]{3}").all()

assert prototype_rfm["ReferenceDate"].nunique() == 1
assert (prototype_rfm["LastPurchaseDate"] <= prototype_rfm["ReferenceDate"]).all()


ALLOWED_SEGMENTS = {
    "Champions",
    "Loyal Customers",
    "At Risk",
    "Potential Loyalists",
    "Hibernating",
    "Others",
}

assert set(prototype_rfm["Segment"]).issubset(ALLOWED_SEGMENTS)
print("Designed customer_segments.csv schema validation passed.")

## 7. Readiness for `cleaned_retail.csv`

Issue #5 only requires the implementation to be ready for the later official dataset.

The helper below demonstrates the intended interface. It is deliberately **not called** in this prototype, and there is no `to_csv(...)`.

In [ ]:
def run_rfm_on_cleaned_csv(csv_path: str | Path):
    cleaned = pd.read_csv(
        csv_path,
        dtype={"InvoiceNo": "string", "CustomerID": "string"},
        low_memory=False,
    )
    return calculate_rfm(cleaned, return_metadata=True)

# Later official-analysis usage:
#
# customer_segments, production_metadata = run_rfm_on_cleaned_csv(
#     PROJECT_ROOT / "data" / "processed" / "cleaned_retail.csv"
# )
#
# Issue #5 does not export customer_segments.csv.

## 8. Issue #5 acceptance checklist

- [x] RFM definitions and reference-date rule are explicit.
- [x] Cancellation, missing/blank InvoiceNo, missing/anonymous customer and invalid-value rules are explicit.
- [x] Duplicate ownership is explicit: the RFM layer audits but does not silently deduplicate.
- [x] R/F/M scoring uses one consistent percentile-rank method from 1–5.
- [x] Equal/low-cardinality values are handled deterministically without `qcut`.
- [x] Six segment labels and their precedence are defined.
- [x] `customer_segments.csv` structure is designed and programmatically validated.
- [x] Pandas prototype runs on deterministic sample data.
- [x] One/multiple orders, multi-line invoice and equal-RFM cases are checked.
- [x] Reusable code is ready to run later on `cleaned_retail.csv`.
- [x] Official full-data analysis and CSV export remain outside Issue #5 scope.